# Notebook 05: Feature Engineering

**Project:** Credit Decision Assistant

## What this notebook is for

Notebook 04 *observed* the data. This notebook *acts* on what it found and produces the final inputs for every model:

1. **Record the decision** to leave income verification out of the risk model.
2. **Fix placeholder values** (9,999,999 means "unknown").
3. **Build affordability features**, since loan amount on its own turned out to be almost useless.
4. **Cap extreme values**, add **missing-value flags**, and **group small categories**.
5. **Bin every feature for the scorecard** using Weight of Evidence, and measure each feature's strength with Information Value.
6. **Choose the scorecard features**: one per family of near-duplicates.
7. **Check whether feature distributions shift** in later years.
8. **Save** everything the modelling notebooks need.

## One rule throughout: what is learned, and from where

Two kinds of change happen here:
- **Fixed rules** use nothing learned from the data ("9,999,999 means missing", "loan-to-income = loan / income"). They're applied the same way to all three splits.
- **Learned settings** (where to cap values, where to draw bin edges, which features to keep) are calculated from the **training set only**, then applied unchanged to validation and test. If the test set helped choose them, its score would no longer be an honest preview of real-life performance.

## A few terms used in this notebook

| Term | What it means here |
|---|---|
| **Capping** | Pulling extreme values down to a set limit, so a handful of huge numbers can't distort a model. |
| **Missing-value flag** | A yes/no column recording that a value was originally missing, so the information isn't lost if the value is later filled in. |
| **Binning** | Cutting a feature into a small number of ranges ("bins"), such as debt-to-income 0% to 16%, 16% to 20%, and so on. |
| **Weight of Evidence (WoE)** | A number for each bin saying how much safer (positive) or riskier (negative) it is than average. |
| **Information Value (IV)** | One number per feature summarising how strongly its bins separate defaulters from non-defaulters. |
| **Monotone** | Steadily going one way: each bin is riskier (or safer) than the one before. |
| **PSI** | Population Stability Index: how much a feature's distribution has shifted between two periods. |

## What this notebook produces

- `data/processed/{train,valid,test}_woe.parquet`: the 15 scorecard features as WoE values, plus IDs and targets.
- `data/processed/{train,valid,test}_features.parquet`: all 77 engineered model features (missing values kept), for gradient boosting and neural networks.
- `models/woe_binners.pkl`: the fitted bins, needed to score new applications.
- `outputs/tables/05_information_value.csv` and `05_psi.csv`.
- An updated config with every rule and setting, so a new application can be processed exactly the same way.

## Step 1: Setup, and record the income-verification decision

Notebook 04 found that borrowers whose income Lending Club verified default *more* often, because Lending Club mainly verified applicants it already considered risky. The field therefore reflects the lender's own suspicion rather than a fact about the borrower. And in this project, "verify income" is one of the conditions the decision engine will set for riskier applicants, so using it as an input to the risk score would make the system feed back on itself.

**Decision:** `verification_status` is excluded from the risk model. The reason is written into the config file so it travels with the project.

The base list of numeric features is read from the full feature list rather than from a separately saved numeric list, so the notebook gives the same result however many times it's run.

**Result:** 61 features remain (59 numeric, 2 categorical: `home_ownership` and `purpose`).

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT  = Path.cwd().parent
INTERIM_DIR   = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIG_DIR       = PROJECT_ROOT / "outputs" / "figures"
TABLE_DIR     = PROJECT_ROOT / "outputs" / "tables"
MODEL_DIR     = PROJECT_ROOT / "models"

CONFIG_PATH = INTERIM_DIR / "audit_config.json"
with open(CONFIG_PATH) as f:
    config = json.load(f)
TARGET = config["target"]["name"]

# Decision from Notebook 04: verification status reflects Lending Club's own suspicion,
# and "verify income" will be a condition set by the decision engine, so it is not a risk input
config["scope"]["exclude_process_fields"] = ["verification_status"]
config["notes"]["verification_status"] = (
    "Excluded from the risk model: Lending Club verified income mainly for applicants it already "
    "considered risky, so the field reflects the lender's judgement, and income verification is a "
    "condition the decision engine sets rather than an input to the risk score."
)
for key in ("all", "categorical"):
    config["features"][key] = [c for c in config["features"][key] if c != "verification_status"]
with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)

# Base numeric list, rebuilt from the full feature list so reruns never pick up
# the new features this notebook adds later
NUMERIC     = [c for c in config["features"]["all"] if c not in config["features"]["categorical"]]
CATEGORICAL = config["features"]["categorical"]
print(f"Features after exclusion: {len(config['features']['all'])} "
      f"({len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical: {CATEGORICAL})")

Features after exclusion: 61 (59 numeric, 2 categorical: ['home_ownership', 'purpose'])


## Step 2: Load all three splits and fix placeholder values

Notebook 04 found **9,999,999** in two credit-limit columns, about 13 and 65 times higher than their 99th percentiles. That's a code meaning "unknown", not a real $10 million credit limit, so it's set to missing. This is a fixed rule, so it's applied to all three splits.

**Result:** fixed for 9 loans in train (6 + 3), 5 in validation and 2 in test.

In [2]:
# All three splits are loaded: fixed rules apply to all of them, learned settings come from train only
splits = {name: pd.read_parquet(PROCESSED_DIR / f"{name}.parquet") for name in ["train", "valid", "test"]}
for name, d in splits.items():
    print(f"{name:<6} {len(d):>9,} loans | default rate {d[TARGET].mean():.2%}")

# Fixed rule (not learned from data): 9,999,999 is a placeholder for "unknown", so treat it as missing
PLACEHOLDER = 9_999_999
PLACEHOLDER_COLS = ["tot_hi_cred_lim", "total_rev_hi_lim"]
for name, d in splits.items():
    for c in PLACEHOLDER_COLS:
        n = (d[c] == PLACEHOLDER).sum()
        d.loc[d[c] == PLACEHOLDER, c] = np.nan
        if n:
            print(f"  {name}: {c} placeholder -> missing for {n} loans")

train    791,027 loans | default rate 3.11%
valid    425,618 loans | default rate 4.01%
test     492,687 loans | default rate 3.74%
  train: tot_hi_cred_lim placeholder -> missing for 6 loans
  train: total_rev_hi_lim placeholder -> missing for 3 loans
  valid: tot_hi_cred_lim placeholder -> missing for 5 loans
  test: tot_hi_cred_lim placeholder -> missing for 2 loans


## Step 3: Affordability features

Notebook 04 showed that loan amount alone is almost useless (Gini 0.005): a $20,000 loan means something very different to someone earning $40,000 than to someone earning $200,000. So these features relate the loan to the borrower's income.

**Loan-to-income:**

$$\text{loan-to-income} = \frac{\text{loan amount}}{\text{annual income}}$$

**A standard monthly payment.** Lending Club's interest rate is excluded (it's Lending Club's own verdict), so a "standard" monthly payment is calculated from the amount and the term using one fixed rate for everyone, 12% a year. It's a common yardstick, not a price. The usual loan payment formula is:

$$\text{payment} = P \cdot \frac{r(1+r)^n}{(1+r)^n - 1}$$

where $P$ is the loan amount, $n$ the number of monthly payments, and $r$ the monthly rate (0.12 / 12 = 0.01).

**Debt-to-income after this loan.** `dti` is the share of monthly income already going on existing debt payments (excluding a mortgage). Adding the new loan's payment shows what the borrower's budget would look like if the loan were approved, which is the core affordability question a lender asks:

$$\text{post-loan DTI} = \text{dti} + 100 \times \frac{\text{payment}}{\text{monthly income}}$$

**Payment-to-income** and **revolving balance-to-income** (credit card debt relative to income) are built too, and every new feature is compared with the raw columns it came from, using the same Gini measure as Notebook 04 on the training set.

**Result:**

| Feature | Gini | Verdict |
|---|---|---|
| post_loan_dti (new) | 0.147 | Stronger than plain `dti` (0.139). Kept |
| loan_to_income (new) | 0.113 | Over 20 times stronger than `loan_amnt` (0.005). Kept |
| payment_to_income (new) | 0.074 | Weaker than expected. Kept for gradient boosting only |
| ref_payment (new) | 0.035, and backwards | Dropped |
| revol_bal_to_income (new) | 0.008 | Dropped |

**Why is payment-to-income weaker than loan-to-income?** Because of the loan term. For the same amount, a 60-month loan has a *smaller* monthly payment than a 36-month one, since it's spread over more months. But 60-month loans are *riskier* (Notebook 02). So the payment ratio makes 60-month loans look more affordable when they are actually riskier, and the two effects partly cancel out. The same reason explains why the standard payment on its own works backwards. Keeping term as its own feature and using loan-to-income for the amount avoids the problem.

**Lending Club's rules show up again:** loan-to-income stops at exactly 0.50 and payment-to-income at about 0.20. Lending Club evidently capped loans at half of annual income, so the data can't tell us what happens beyond that. That's worth remembering for the decision engine.

In [3]:
REF_ANNUAL_RATE = 0.12                 # one fixed rate for everyone: a common yardstick, not a price
r = REF_ANNUAL_RATE / 12               # monthly rate

def add_affordability(d: pd.DataFrame) -> pd.DataFrame:
    n = d["term_m"].astype("float64")                       # number of monthly payments
    growth = (1 + r) ** n
    payment = d["loan_amnt"] * r * growth / (growth - 1)    # standard loan payment formula
    monthly_income = d["annual_inc"] / 12

    d["ref_payment"]          = payment.astype("float32")
    d["loan_to_income"]       = (d["loan_amnt"] / d["annual_inc"]).astype("float32")
    d["payment_to_income"]    = (payment / monthly_income).astype("float32")
    d["post_loan_dti"]        = (d["dti"] + 100 * payment / monthly_income).astype("float32")
    d["revol_bal_to_income"]  = (d["revol_bal"] / d["annual_inc"]).astype("float32")
    return d

for name in splits:
    splits[name] = add_affordability(splits[name])

NEW_FEATURES = ["ref_payment", "loan_to_income", "payment_to_income", "post_loan_dti", "revol_bal_to_income"]

# How do the new features compare with the raw columns they were built from? (train only)
train = splits["train"]
def gini(col):
    present = train[col].notna()
    return 2 * roc_auc_score(train.loc[present, TARGET], train.loc[present, col]) - 1

compare = pd.DataFrame({"signed_gini": {c: gini(c) for c in
                        NEW_FEATURES + ["loan_amnt", "annual_inc", "dti", "revol_bal", "term_m"]}})
compare["gini"] = compare["signed_gini"].abs()
compare["type"] = np.where(compare.index.isin(NEW_FEATURES), "new", "raw")
display(compare.sort_values("gini", ascending=False)
               .style.format({"signed_gini": "{:+.3f}", "gini": "{:.3f}"})
               .bar(subset=["gini"], color="#9AA5B1"))
print(train[NEW_FEATURES].describe(percentiles=[.01, .5, .99]).T.round(3))

,signed_gini,gini,type
post_loan_dti,+0.147,0.147,new
dti,+0.139,0.139,raw
loan_to_income,+0.113,0.113,new
annual_inc,-0.110,0.110,raw
term_m,+0.090,0.090,raw
payment_to_income,+0.074,0.074,new
revol_bal,-0.061,0.061,raw
ref_payment,-0.035,0.035,new
revol_bal_to_income,+0.008,0.008,new
loan_amnt,+0.005,0.005,raw


                           count     mean      std     min      1%      50%        99%        max
ref_payment         791,027.0000 430.7350 235.7810 22.2440 66.4290 381.9650 1,162.5010 1,162.5010
loan_to_income      791,027.0000   0.2240   0.1110  0.0000  0.0300   0.2120     0.4800     0.5000
payment_to_income   791,027.0000   0.0780   0.0380  0.0000  0.0120   0.0730     0.1790     0.1990
post_loan_dti       791,027.0000  26.2500   9.8980  0.0620  6.4660  25.8060    48.8240    59.8490
revol_bal_to_income 791,027.0000   0.2400   0.2100  0.0000  0.0070   0.1950     0.9400    10.2180


## Step 4: Drop weak new features, cap extremes, flag missing values, group categories

**1. Drop** `ref_payment` and `revol_bal_to_income` (Step 3).

**2. Cap extreme values.** For each numeric feature, find the value that 99.5% of *training* loans fall below, and pull anything higher down to it. Very large values can drag a scorecard or neural network around, while gradient boosting and binning only care about order, so capping helps the first two and costs the others nothing. The caps are learned on train and saved, so validation, test and future applications get exactly the same treatment.

**3. Missing-value flags.** For every feature with at least 0.5% missing in train, a yes/no column `<feature>_missing` is added, and the original column keeps its missing values. Gradient boosting handles missing values itself, and the scorecard bins give "missing" its own bin. The neural network will need missing values filled in later, and the flag preserves the information that the value was originally missing. That matters because Notebook 04 showed missing can mean the safest group (no recent credit check) or the riskiest (no credit card).

**4. Group small categories.**
- `home_ownership` "OTHER" has only 3 training loans, too few to learn from. It's merged into RENT, the riskiest of the large groups, since being cautious about an unknown group is the safer mistake for a lender.
- `purpose` categories with fewer than 2,000 training loans are merged into "other". Small but clearly different purposes (moving, medical, small business, each above 4,000 loans) stay separate.

**Result:**
- **57 features capped.** For example, income at $310,000 (so the $9.5 million value becomes $310,000), revolving utilisation at 99.9% (anything above 100% just means "maxed out"), and credit history at 518 months (about 43 years). Even count features have a long tail of a few unusually large values, which is why so many got a cap; only the top 0.5% of each feature is affected.
- **13 missing-value flags.** Four of them (`bc_open_to_buy`, `bc_util`, `percent_bc_gt_75`, `mths_since_recent_bc`) are missing for the same roughly 1% of borrowers, the people with no credit card, so they're near-identical. That doesn't matter for the scorecard or gradient boosting, and only one will be kept for the neural network.
- **Purposes merged into "other":** wedding, renewable energy and educational. Home ownership is down to MORTGAGE, OWN and RENT.

In [4]:
# 1. Drop the new features that didn't earn their place
DROP_NEW = ["ref_payment", "revol_bal_to_income"]
for name in splits:
    splits[name] = splits[name].drop(columns=DROP_NEW)
NEW_FEATURES = [c for c in NEW_FEATURES if c not in DROP_NEW]
NUMERIC = NUMERIC + NEW_FEATURES
train = splits["train"]

# 2. Cap extreme values at the training set's 99.5th percentile (learned on train only)
CAP_Q = 0.995
caps = {}
for c in NUMERIC:
    if c == "term_m":
        continue                                # only two values, nothing to cap
    cap = float(train[c].quantile(CAP_Q))
    n_above = int((train[c] > cap).sum())
    if n_above > 0:
        caps[c] = cap
for name, d in splits.items():
    for c, cap in caps.items():
        d[c] = d[c].clip(upper=cap)
print(f"Capped {len(caps)} features at the train {CAP_Q:.1%} percentile. Examples:")
for c in ["annual_inc", "revol_bal", "tot_cur_bal", "revol_util", "credit_history_m"]:
    if c in caps:
        print(f"  {c:<20} cap = {caps[c]:,.1f}")

# 3. Missing-value flags for features with at least 0.5% missing in train
miss_share = train[NUMERIC].isna().mean()
FLAG_COLS = miss_share[miss_share >= 0.005].index.tolist()
for name, d in splits.items():
    for c in FLAG_COLS:
        d[f"{c}_missing"] = d[c].isna().astype("int8")
MISSING_FLAGS = [f"{c}_missing" for c in FLAG_COLS]
print(f"\nAdded {len(MISSING_FLAGS)} missing-value flags:", FLAG_COLS)

# 4. Group small categories (rules decided from train)
purpose_counts = train["purpose"].value_counts()
SMALL_PURPOSES = purpose_counts[purpose_counts < 2_000].index.tolist()
for name, d in splits.items():
    d["home_ownership"] = d["home_ownership"].replace({"OTHER": "RENT"})
    d["purpose"] = d["purpose"].where(~d["purpose"].isin(SMALL_PURPOSES), "other")
print(f"\nPurposes merged into 'other': {SMALL_PURPOSES}")
for col in ["home_ownership", "purpose"]:
    print(f"{col}: {sorted(train[col].unique())}")

Capped 57 features at the train 99.5% percentile. Examples:
  annual_inc           cap = 310,000.0
  revol_bal            cap = 135,785.0
  tot_cur_bal          cap = 786,255.9
  revol_util           cap = 99.9
  credit_history_m     cap = 518.0

Added 13 missing-value flags: ['mths_since_last_delinq', 'mths_since_last_record', 'mths_since_last_major_derog', 'bc_open_to_buy', 'bc_util', 'mo_sin_old_il_acct', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq', 'mths_since_recent_inq', 'mths_since_recent_revol_delinq', 'num_tl_120dpd_2m', 'percent_bc_gt_75', 'emp_length_yrs']

Purposes merged into 'other': ['wedding', 'renewable_energy', 'educational']
home_ownership: ['MORTGAGE', 'OWN', 'RENT']
purpose: ['car', 'credit_card', 'debt_consolidation', 'home_improvement', 'house', 'major_purchase', 'medical', 'moving', 'other', 'small_business', 'vacation']


## Step 5: Bin every feature and measure its strength

**Why bins?** A scorecard is a logistic regression: it adds up a weighted number for each feature. That assumes each feature affects risk in a straight line. Notebook 04 showed this isn't always true (debt-to-income bends sharply upward, and missing values can mean "safest" or "riskiest"). So each feature is cut into a few bins, and each value is replaced by a number describing how risky its bin is. That lets a simple, readable model capture curves and handle missing values sensibly.

**Weight of Evidence (WoE).** For each bin, compare its share of all the non-defaulters with its share of all the defaulters:

$$\text{WoE}_{\text{bin}} = \ln\left(\frac{\%\text{ of all non-defaulters in this bin}}{\%\text{ of all defaulters in this bin}}\right)$$

A bin holding 10% of the good loans but only 5% of the defaults is safe: WoE = ln(2) = +0.69. One holding 10% of the good loans but 20% of the defaults is risky: WoE = ln(0.5) = -0.69. An exactly average bin gets 0. WoE sits on the same scale as logistic regression (log-odds), which is why scorecards pair the two.

**Information Value (IV)** adds this up across a feature's bins:

$$\text{IV} = \sum_{\text{bins}} \left(\%\text{ non-defaulters}_{\text{bin}} - \%\text{ defaulters}_{\text{bin}}\right) \times \text{WoE}_{\text{bin}}$$

Rules of thumb: below 0.02 useless, 0.02 to 0.10 weak, 0.10 to 0.30 medium, above 0.30 strong, above 0.50 suspicious (worth checking for leakage).

**How the bins are chosen.** The `optbinning` library finds the best edges by optimisation, under these rules:
- **Monotone:** bins must step steadily up or down in risk. That matches Notebook 04, stops the bins chasing noise, and keeps the scorecard sensible (more debt can never earn more points).
- **Each bin holds at least 2% of training loans** (about 15,800), so every WoE value rests on plenty of data.
- **At most 10 bins** per feature.
- **Missing values get their own bin.**
- **The outer bins are open-ended**, so values outside the training range, like the larger loans issued from 2016, still fall into a bin.

**Result:** of 64 features, **2 are medium, 23 weak and 39 useless.** That's typical for credit bureau data: no single feature is strong, and the predictive power comes from combining many modest signals.
- The two "medium" features are accounts opened in the last 24 months (0.111) and 12 months (0.105). Credit hunger leads again.
- `post_loan_dti` (0.068) beats `dti` (0.063), and `loan_to_income` (0.043) is far above `loan_amnt` (0.002).
- `purpose` was grouped into 4 risk bins (0.046), and `term` scores 0.036. Two-value features can't score highly on these measures but still add information.
- All the past late payment and public record features are useless here. Several got just one bin, meaning no split where the default rate genuinely differs, which confirms the selection-bias finding from Notebook 03.
- `payment_to_income` (0.019) falls just below the cut-off, as Step 3 suggested.

The IVs rank the features in almost the same order as the Gini values in Notebook 04, so two different measures agree.

In [5]:
import warnings
from optbinning import OptimalBinning

warnings.filterwarnings("ignore", category=FutureWarning)

BIN_FEATURES = NUMERIC + CATEGORICAL            # numeric (incl. new ratios) + home_ownership, purpose
# Bins are learned on the training set only
y_train = train[TARGET].values

binners, rows = {}, []
for c in BIN_FEATURES:
    is_cat = c in CATEGORICAL
    params = dict(name=c, dtype="categorical" if is_cat else "numerical",
                  max_n_bins=10, min_bin_size=0.02, solver="cp")
    if not is_cat:
        params["monotonic_trend"] = "auto_asc_desc"     # bins must step steadily up or steadily down
    ob = OptimalBinning(**params)
    ob.fit(train[c].values, y_train)
    table = ob.binning_table.build()
    binners[c] = ob
    rows.append({
        "feature":       c,
        "iv":            table.loc["Totals", "IV"],
        "n_bins":        len(table) - 3,                 # minus the Special, Missing and Totals rows
        "missing_share": train[c].isna().mean(),
    })

iv = pd.DataFrame(rows).set_index("feature").sort_values("iv", ascending=False)

def iv_label(v):
    return ("useless" if v < 0.02 else "weak" if v < 0.10 else
            "medium" if v < 0.30 else "strong" if v < 0.50 else "suspicious")
iv["strength"] = iv["iv"].map(iv_label)
iv.to_csv(TABLE_DIR / "05_information_value.csv")

print(iv["strength"].value_counts().to_string())
display(iv.style.format({"iv": "{:.4f}", "missing_share": "{:.1%}"})
             .bar(subset=["iv"], color="#9AA5B1"))

strength
useless    39
weak       23
medium      2


,iv,n_bins,missing_share,strength
feature,,,,
acc_open_past_24mths,0.1111,9,0.0%,medium
num_tl_op_past_12m,0.1051,6,0.0%,medium
fico_range_low,0.0907,10,0.0%,weak
mths_since_recent_inq,0.0745,10,10.2%,weak
mo_sin_rcnt_tl,0.0715,10,0.0%,weak
inq_last_6mths,0.0683,4,0.0%,weak
post_loan_dti,0.0675,10,0.0%,weak
dti,0.0625,10,0.0%,weak
mo_sin_old_rev_tl_op,0.0495,10,0.0%,weak


### A look inside the bins

Five features, bin by bin. "Event rate" is the default rate in that bin. optbinning always adds a **Special** row (for special codes, unused here) and a **Missing** row. The table's final "Totals" row is left out because it holds text in some columns.

**Result:**
- **Every numeric feature steps steadily in one direction.** `post_loan_dti` runs from 2.10% (below 16.3%, WoE +0.40) to 4.80% (43% and above, WoE -0.45). FICO runs from 4.32% (below 662.5, WoE -0.34) to 1.45% (748 and above, WoE +0.78). Accounts opened in 24 months run from 1.75% (0 or 1 account, WoE +0.59) to 5.44% (10 or more, WoE -0.58).
- **For `mths_since_recent_inq`, the Missing bin is the safest of all:** 10.2% of loans with a 1.90% default rate and WoE +0.506, higher than any real value. No recent credit check at all is the best case, exactly as Notebook 04 suggested. A model fed raw numbers with averages filled in could never learn this.
- **`purpose` was grouped into four risk tiers:** credit card refinancing (2.21%); car and home improvement (3.00%); debt consolidation and major purchase (3.27%); and vacation, other, house, medical, small business and moving (4.73%). Vacation landing in the riskiest tier is plausible, since borrowing for a holiday is discretionary spending on credit.
- **Features with no missing values in train have an empty Missing bin with WoE 0.** If a value is ever missing later, it's treated as average risk, a neutral and cautious default.

In [7]:
for c in ["post_loan_dti", "fico_range_low", "acc_open_past_24mths", "mths_since_recent_inq", "purpose"]:
    table = binners[c].binning_table.build()
    table = table.drop(index="Totals")          # the Totals row holds text in some columns, so leave it out
    print(f"\n{c}  (IV {iv.loc[c, 'iv']:.4f})")
    display(table[["Bin", "Count", "Count (%)", "Event rate", "WoE", "IV"]]
              .style.format({"Count": "{:,.0f}", "Count (%)": "{:.1%}", "Event rate": "{:.2%}",
                             "WoE": "{:+.3f}", "IV": "{:.4f}"}))


post_loan_dti  (IV 0.0675)


,Bin,Count,Count (%),Event rate,WoE,IV
0,"(-inf, 16.29)","134,708",17.0%,2.10%,+0.403,0.0230
1,"[16.29, 20.01)","93,571",11.8%,2.40%,+0.266,0.0074
2,"[20.01, 21.44)","40,007",5.1%,2.52%,+0.217,0.0022
3,"[21.44, 25.70)","124,039",15.7%,2.87%,+0.082,0.0010
4,"[25.70, 28.26)","73,221",9.3%,3.05%,+0.020,0.0000
5,"[28.26, 30.10)","49,976",6.3%,3.22%,-0.034,0.0001
6,"[30.10, 34.41)","102,729",13.0%,3.58%,-0.146,0.0030
7,"[34.41, 37.81)","64,387",8.1%,3.93%,-0.242,0.0054
8,"[37.81, 43.08)","68,118",8.6%,4.36%,-0.351,0.0125
9,"[43.08, inf)","40,271",5.1%,4.80%,-0.452,0.0129



fico_range_low  (IV 0.0907)


,Bin,Count,Count (%),Event rate,WoE,IV
0,"(-inf, 662.50)","74,739",9.4%,4.32%,-0.341,0.0129
1,"[662.50, 667.50)","72,752",9.2%,4.00%,-0.260,0.0070
2,"[667.50, 677.50)","137,709",17.4%,3.78%,-0.201,0.0077
3,"[677.50, 687.50)","119,190",15.1%,3.49%,-0.118,0.0022
4,"[687.50, 692.50)","54,061",6.8%,3.14%,-0.010,0.0000
5,"[692.50, 697.50)","48,472",6.1%,2.95%,+0.056,0.0002
6,"[697.50, 712.50)","116,170",14.7%,2.55%,+0.205,0.0056
7,"[712.50, 722.50)","53,937",6.8%,2.07%,+0.420,0.0099
8,"[722.50, 747.50)","66,363",8.4%,1.84%,+0.541,0.0192
9,"[747.50, inf)","47,634",6.0%,1.45%,+0.780,0.0259



acc_open_past_24mths  (IV 0.1111)


,Bin,Count,Count (%),Event rate,WoE,IV
0,"(-inf, 1.50)","112,895",14.3%,1.75%,+0.587,0.0377
1,"[1.50, 2.50)","108,382",13.7%,2.21%,+0.351,0.0144
2,"[2.50, 3.50)","119,889",15.2%,2.65%,+0.163,0.0038
3,"[3.50, 4.50)","112,771",14.3%,2.95%,+0.055,0.0004
4,"[4.50, 5.50)","95,553",12.1%,3.32%,-0.066,0.0005
5,"[5.50, 6.50)","74,465",9.4%,3.64%,-0.163,0.0027
6,"[6.50, 7.50)","54,909",6.9%,4.02%,-0.267,0.0056
7,"[7.50, 9.50)","63,546",8.0%,4.71%,-0.431,0.0184
8,"[9.50, inf)","48,617",6.1%,5.44%,-0.582,0.0276
9,Special,0,0.0%,0.00%,+0.000,0.0000



mths_since_recent_inq  (IV 0.0745)


,Bin,Count,Count (%),Event rate,WoE,IV
0,"(-inf, 0.50)","63,641",8.0%,4.65%,-0.417,0.0171
1,"[0.50, 1.50)","80,437",10.2%,4.26%,-0.326,0.0126
2,"[1.50, 2.50)","63,336",8.0%,3.77%,-0.199,0.0035
3,"[2.50, 3.50)","57,489",7.3%,3.60%,-0.152,0.0018
4,"[3.50, 5.50)","98,293",12.4%,3.24%,-0.041,0.0002
5,"[5.50, 7.50)","81,440",10.3%,3.04%,+0.023,0.0001
6,"[7.50, 10.50)","92,284",11.7%,2.68%,+0.155,0.0026
7,"[10.50, 12.50)","45,773",5.8%,2.53%,+0.213,0.0024
8,"[12.50, 16.50)","63,963",8.1%,2.42%,+0.258,0.0048
9,"[16.50, inf)","63,537",8.0%,2.20%,+0.357,0.0087



purpose  (IV 0.0455)


,Bin,Count,Count (%),Event rate,WoE,IV
0,"['credit_card'] Length: 1, dtype: str","190,237",24.0%,2.21%,+0.353,0.0255
1,"['car', 'home_improvement'] Length: 2, dtype: str","52,063",6.6%,3.00%,+0.039,0.0001
2,"['debt_consolidation', 'major_purchase'] Length: 2, dtype: str","486,936",61.6%,3.27%,-0.052,0.0017
3,"['vacation', 'other', 'house', 'medical', 'small_business', 'moving'] Length: 6, dtype: str","61,791",7.8%,4.73%,-0.436,0.0183
4,Special,0,0.0%,0.00%,+0.000,0.0000
5,Missing,0,0.0%,0.00%,+0.000,0.0000


## Step 6: Choose the scorecard features

**Filter 1: IV of at least 0.02.** That keeps the 25 weak and medium features.

**Filter 2: one feature per family.** Several survivors carry nearly the same information. A scorecard handles that badly: it can't tell which feature deserves the credit, so their weights become unstable and can even flip sign. Instead of choosing by hand, a simple, repeatable rule is used:
1. Turn each candidate into its WoE values (what the scorecard actually uses).
2. Go through the candidates from highest IV to lowest.
3. Keep a feature only if its correlation with every feature already kept is below 0.6. Otherwise drop it and record which stronger feature it overlaps with.

Going from strongest to weakest means the most predictive member of each family survives.

**Result: 15 features selected**, with the largest remaining correlation at 0.599. They cover distinct aspects of a borrower:

| Aspect | Features |
|---|---|
| Credit hunger | acc_open_past_24mths, mths_since_recent_inq, mo_sin_rcnt_tl |
| Credit score | fico_range_low |
| Affordability | post_loan_dti, loan_to_income, annual_inc |
| Depth of credit history | mo_sin_old_rev_tl_op, mo_sin_old_il_acct, mths_since_recent_bc |
| Available credit | tot_hi_cred_lim, bc_open_to_buy |
| The loan itself | purpose, term_m |
| Housing | home_ownership |

**The 10 dropped features each lost to a stronger relative**: `dti` to `post_loan_dti` (0.896), `num_tl_op_past_12m` to `acc_open_past_24mths` (0.728), `credit_history_m` to `mo_sin_old_rev_tl_op` (0.874), `tot_cur_bal` to `tot_hi_cred_lim` (0.968), and so on. One less obvious case: `mort_acc` (number of mortgages) overlaps `tot_hi_cred_lim` (0.629), because a mortgage is usually someone's largest credit line. `annual_inc` and `loan_to_income` both survive: income on its own and the loan relative to income say different things.

In [8]:
IV_MIN = 0.02           # below this a feature carries essentially no signal
CORR_MAX = 0.6          # above this two features say largely the same thing

candidates = iv[iv["iv"] >= IV_MIN].index.tolist()      # already sorted from highest IV to lowest
print(f"Features with IV >= {IV_MIN}: {len(candidates)}")

# Convert each candidate to its WoE values on the training set
woe_train = pd.DataFrame({c: binners[c].transform(train[c].values, metric="woe") for c in candidates},
                         index=train.index)
woe_corr = woe_train.corr()

selected, dropped = [], []
for c in candidates:
    clashes = [s for s in selected if abs(woe_corr.loc[c, s]) >= CORR_MAX]
    if clashes:
        worst = max(clashes, key=lambda s: abs(woe_corr.loc[c, s]))
        dropped.append({"feature": c, "iv": iv.loc[c, "iv"], "overlaps_with": worst,
                        "corr": woe_corr.loc[c, worst]})
    else:
        selected.append(c)

print(f"\nSelected for the scorecard: {len(selected)}")
display(iv.loc[selected, ["iv", "n_bins", "strength"]].style.format({"iv": "{:.4f}"}))
print(f"\nDropped as overlapping with a stronger feature: {len(dropped)}")
display(pd.DataFrame(dropped).style.format({"iv": "{:.4f}", "corr": "{:+.3f}"}))

# Largest correlation left between selected features (should be below CORR_MAX)
sel_corr = woe_corr.loc[selected, selected].abs().where(~np.eye(len(selected), dtype=bool))
print(f"\nHighest remaining correlation between selected features: {sel_corr.max().max():.3f}")

Features with IV >= 0.02: 25

Selected for the scorecard: 15


,iv,n_bins,strength
feature,,,
acc_open_past_24mths,0.1111,9,medium
fico_range_low,0.0907,10,weak
mths_since_recent_inq,0.0745,10,weak
mo_sin_rcnt_tl,0.0715,10,weak
post_loan_dti,0.0675,10,weak
mo_sin_old_rev_tl_op,0.0495,10,weak
bc_open_to_buy,0.0465,10,weak
purpose,0.0455,4,weak
tot_hi_cred_lim,0.0455,10,weak



Dropped as overlapping with a stronger feature: 10


,feature,iv,overlaps_with,corr
0,num_tl_op_past_12m,0.1051,acc_open_past_24mths,+0.728
1,inq_last_6mths,0.0683,mths_since_recent_inq,+0.662
2,dti,0.0625,post_loan_dti,+0.896
3,credit_history_m,0.0414,mo_sin_old_rev_tl_op,+0.874
4,total_bc_limit,0.0400,bc_open_to_buy,+0.736
5,avg_cur_bal,0.0397,tot_hi_cred_lim,+0.847
6,mo_sin_rcnt_rev_tl_op,0.0372,mo_sin_rcnt_tl,+0.709
7,tot_cur_bal,0.0365,tot_hi_cred_lim,+0.968
8,mort_acc,0.0324,tot_hi_cred_lim,+0.629
9,total_rev_hi_lim,0.0312,bc_open_to_buy,+0.652



Highest remaining correlation between selected features: 0.599


## Step 7: Are the features stable in later years?

The model learns from 2013 to 2015 and will be used on later loans. If a feature's distribution shifts a lot, the model is working outside what it learned. The standard check in lending is the **Population Stability Index**. For each feature, compare the share of loans in each bin between training and a later period:

$$\text{PSI} = \sum_{\text{bins}} \left(\text{share}_{\text{later}} - \text{share}_{\text{train}}\right) \times \ln\frac{\text{share}_{\text{later}}}{\text{share}_{\text{train}}}$$

Below 0.10 is stable, 0.10 to 0.25 a moderate shift worth watching, above 0.25 a major shift. A tiny number is added to every share so an empty bin doesn't break the logarithm.

This check uses no outcomes from validation or test, only how the feature values are spread out. It says nothing about how well the model will perform; it flags features to keep an eye on, so it doesn't compromise the test set.

**Result:** **61 stable, 3 moderate, 0 major.**
- The three moderate shifts are one family, **credit card usage** (`percent_bc_gt_75` 0.129, `bc_util` 0.111, `bc_open_to_buy` 0.107). All three are stable in 2016 and only cross 0.10 in 2017 and 2018, which fits Lending Club's shift towards safer borrowers with more unused card limit from 2016. Only `bc_open_to_buy` is in the scorecard. It's kept and flagged to watch.
- The scorecard's most important features barely move: credit hunger (0.003 to 0.004), `post_loan_dti` (0.017) and income (0.016). `loan_to_income` moves only a little (0.057) despite the 2016 loan cap increase, because the open-ended top bin absorbs the larger loans.

The strongest features are also the steadiest.

In [9]:
import pickle                         # used in the next cell to save the fitted bins

# Cell 8: Population Stability Index for every binned feature, train vs validation and vs test
def psi(train_share, other_share, eps=1e-6):
    aligned = pd.concat([train_share, other_share], axis=1).fillna(0) + eps
    a, e = aligned.iloc[:, 1], aligned.iloc[:, 0]
    return float(((a - e) * np.log(a / e)).sum())

def bin_shares(feature, d):
    labels = binners[feature].transform(d[feature].values, metric="bins")   # which bin each loan falls in
    return pd.Series(labels).astype(str).value_counts(normalize=True)

psi_rows = []
for c in BIN_FEATURES:
    base = bin_shares(c, splits["train"])
    psi_rows.append({"feature": c,
                     "psi_valid": psi(base, bin_shares(c, splits["valid"])),
                     "psi_test":  psi(base, bin_shares(c, splits["test"])),
                     "in_scorecard": c in selected})
psi_df = pd.DataFrame(psi_rows).set_index("feature").sort_values("psi_test", ascending=False)
psi_df["status"] = pd.cut(psi_df["psi_test"], [-np.inf, 0.10, 0.25, np.inf],
                          labels=["stable", "moderate shift", "major shift"])
psi_df.to_csv(TABLE_DIR / "05_psi.csv")
print(psi_df["status"].value_counts().to_string())
display(psi_df.head(20).style.format({"psi_valid": "{:.3f}", "psi_test": "{:.3f}"}))
print("\nScorecard features:")
display(psi_df[psi_df["in_scorecard"]].style.format({"psi_valid": "{:.3f}", "psi_test": "{:.3f}"}))

status
stable            61
moderate shift     3
major shift        0


,psi_valid,psi_test,in_scorecard,status
feature,,,,
percent_bc_gt_75,0.041,0.129,False,moderate shift
bc_util,0.033,0.111,False,moderate shift
bc_open_to_buy,0.026,0.107,True,moderate shift
revol_util,0.033,0.098,False,stable
num_rev_accts,0.021,0.064,False,stable
num_bc_tl,0.031,0.063,False,stable
mo_sin_old_rev_tl_op,0.026,0.059,True,stable
loan_to_income,0.025,0.057,True,stable
payment_to_income,0.021,0.056,False,stable



Scorecard features:


,psi_valid,psi_test,in_scorecard,status
feature,,,,
bc_open_to_buy,0.026,0.107,True,moderate shift
mo_sin_old_rev_tl_op,0.026,0.059,True,stable
loan_to_income,0.025,0.057,True,stable
fico_range_low,0.005,0.053,True,stable
purpose,0.020,0.045,True,stable
mo_sin_old_il_acct,0.008,0.022,True,stable
post_loan_dti,0.001,0.017,True,stable
annual_inc,0.007,0.016,True,stable
term_m,0.016,0.010,True,stable


## Step 8: Save everything later notebooks need

- **WoE tables:** the 15 scorecard features as WoE values, plus IDs and targets, for the scorecard notebook. `term_m` is saved as `term_m_raw` so it can't be confused with its WoE version.
- **Feature tables:** all 77 engineered model features with missing values intact, plus the missing flags, for gradient boosting and neural networks.
- **The fitted bins:** needed to turn a new application into WoE values in the app.
- **The config:** caps, grouping rules, the reference rate, the selection thresholds and the feature lists, so a new application can be processed exactly as the training data was.

**Result:** WoE tables with 22 columns (7 ID and target columns + 15 WoE features), and feature tables with 83 columns (7 ID and target columns + 77 model features, with `term_m` stored once). Bins saved for all 64 features.

In [10]:
# Cell 9: save WoE tables, feature tables, fitted bins and settings
SCORECARD_FEATURES = selected
ID_COLS = ["id", "issue_d", "issue_year", "term_m",
           TARGET, "default_12m_lenient", "default_12m_strict"]
MODEL_FEATURES = NUMERIC + CATEGORICAL + MISSING_FLAGS      # for gradient boosting and neural networks

for name, d in splits.items():
    # WoE version of the 15 scorecard features
    woe = pd.DataFrame({f"woe_{c}": binners[c].transform(d[c].values, metric="woe")
                        for c in SCORECARD_FEATURES}, index=d.index).astype("float32")
    woe = pd.concat([d[ID_COLS].rename(columns={"term_m": "term_m_raw"}), woe], axis=1)
    woe.to_parquet(PROCESSED_DIR / f"{name}_woe.parquet", index=False)

    # Full engineered feature table, missing values kept
    keep = list(dict.fromkeys(ID_COLS + MODEL_FEATURES))    # dict.fromkeys removes the duplicate term_m
    d[keep].to_parquet(PROCESSED_DIR / f"{name}_features.parquet", index=False)
    print(f"{name:<6} woe: {woe.shape} | features: {d[keep].shape}")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
with open(MODEL_DIR / "woe_binners.pkl", "wb") as f:
    pickle.dump({c: binners[c] for c in BIN_FEATURES}, f)

config["feature_engineering"] = {
    "placeholder_value": PLACEHOLDER,
    "placeholder_cols": PLACEHOLDER_COLS,
    "reference_annual_rate": REF_ANNUAL_RATE,
    "new_features": NEW_FEATURES,
    "cap_quantile": CAP_Q,
    "caps": caps,
    "missing_flag_cols": FLAG_COLS,
    "home_ownership_map": {"OTHER": "RENT"},
    "small_purposes_to_other": SMALL_PURPOSES,
    "binning": {"max_n_bins": 10, "min_bin_size": 0.02, "monotonic_trend": "auto_asc_desc"},
    "iv_min": IV_MIN,
    "woe_corr_max": CORR_MAX,
}
config["features"]["scorecard"] = SCORECARD_FEATURES
config["features"]["model"] = MODEL_FEATURES
config["features"]["model_numeric"] = NUMERIC          # base numeric features + new ratios
# Keep "numeric" as the base list from Notebook 03 (minus exclusions), so reruns stay consistent
config["features"]["numeric"] = [c for c in config["features"]["all"] if c not in CATEGORICAL]
with open(CONFIG_PATH, "w") as f:
    json.dump(config, f, indent=2)
print(f"\nSaved binners for {len(BIN_FEATURES)} features and updated config "
      f"({len(SCORECARD_FEATURES)} scorecard features, {len(MODEL_FEATURES)} model features)")

train  woe: (791027, 22) | features: (791027, 83)
valid  woe: (425618, 22) | features: (425618, 83)
test   woe: (492687, 22) | features: (492687, 83)

Saved binners for 64 features and updated config (15 scorecard features, 77 model features)


## Findings

1. **Affordability matters more than size.** Debt-to-income after the new loan (Gini 0.147) beats plain debt-to-income, and loan-to-income is over 20 times stronger than loan amount on its own.
2. **A sensible-sounding feature can work against another one.** Payment-to-income makes 60-month loans look more affordable even though they're riskier, so the loan term stays as its own feature instead.
3. **Income verification is excluded** from the risk model because it reflects Lending Club's own suspicion, and verification will be a condition the decision engine sets.
4. **Extreme values are capped, missing values are flagged, and small categories are grouped**, all using rules learned from the training set only.
5. **No single feature is strong:** only 2 of 64 reach "medium" IV and 39 are useless. Past late payments and public records carry no signal among approved borrowers.
6. **Binning captures what a straight line can't.** For example, no recent credit check (a missing value) is the safest group of all.
7. **The scorecard uses 15 features**, one per family, covering credit hunger, credit score, affordability, credit history, available credit, the loan itself and housing.
8. **61 of 64 features are stable over time.** Only credit card usage shifts moderately in 2017 and 2018, and the strongest features barely move.

**Next:** Notebook 06 builds the scorecard: a logistic regression on the 15 WoE features, turned into points a person can read.